In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score
import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "codes" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "codes" / "agents"))
import agent_scorer as scorer

source_dir = str(PROJECT_ROOT / "outputs" / "opt_prompt") + "/"
files = sorted(f for f in os.listdir(source_dir) if f.endswith(".txt"))
# print(files)
import numpy as np
def evaluate_in_chunks(golds, preds, scorer, chunk_size=30000):
    n_chunks  = (len(golds) // chunk_size) + 1
    precisions, recalls, f1s = [], [], []

    for i in range(n_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < n_chunks - 1 else len(golds)  # last chunk takes remainder

        g_chunk = golds[start:end]
        p_chunk = preds[start:end]

        if len(g_chunk) == 0:  # skip empty chunks
            continue

        sc = scorer.score(g_chunk, p_chunk, False)  # returns (P, R, F1)
        precisions.append(sc[0])
        recalls.append(sc[1])
        f1s.append(sc[2])

    precisions, recalls, f1s = np.array(precisions), np.array(recalls), np.array(f1s)

    # for p, r, f1 in zip(precisions, recalls, f1s):
    #     print(f"{p*100:.2f}\t{r*100:.2f}\t{f1*100:.2f}")

    results = {
        "mean_P": np.mean(precisions) * 100,
        "std_P": np.std(precisions) * 100,
        "mean_R": np.mean(recalls) * 100,
        "std_R": np.std(recalls) * 100,
        "mean_F1": np.mean(f1s) * 100,
        "std_F1": np.std(f1s) * 100,
    }

    return results

for f in files:
    print(f)

tacred_qwen_initial_prompt_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_x_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_x_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_x_gradpo-prob_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_x_greater-tg_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_x_greater_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
tacred_qwen_rpo_node_y_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-1

In [14]:
cccnt = 0
for ii, ff in enumerate(files):
    source = ff
    # if not ff.startswith('Qwen') and not ff.startswith("meta-llama") and not ff.startswith("microsoft"):
    # if not (ff.startswith('meta-llama') or "gemma-2-9b-it" in ff or ff.startswith("microsoft")) :
    # if not ("Qwen-Qwen3-4B-5shots_fs_tacred_test_episodes_5shots_softmatch.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20251202_hybrid.txt" in ff) :
    #     continue
    # if "main_test_qwen4b-ps_3-initial_prompt_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260429.txt" not in source:
    #     continue
    if (not source.endswith(".txt")):
        continue

    if "tacred_qwen_rpo_node_y_gradpo" not in source:
        continue
    print(f"{ii} ************************** ")
    cccnt += 1
    print(source)
    p_list = []
    q_list = []
    rl_list = []
    r_list = []

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            p_list.append(line)

        if '@@@@@@@@@@' in line:
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            q_list.append(line)

        if line == '$$$$$ query relation':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            rl_list.append(line)

        if line == '$$$$$ relation list':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            r_list.append(line)

        if line == '$$$$$ votes':
            ready = True
        else:
            ready = False


    for i in range(len(rl_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(rl_list[i])
        rl_list[i] = reconstructed_list

    for i in range(len(r_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(r_list[i])
        r_list[i] = reconstructed_list


    for i in range(len(p_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(p_list[i])
        p_list[i] = reconstructed_list

    # print(p_list[-1][0][0]['content'])

    cnt = 0
    trans = []
    trans_ = []
    for i in range(len(rl_list)):
        dec = None
        dec_ = []
        mul = False

        # ens_res = ens[i]

        for wi, (rrr, yyy) in enumerate (zip(r_list[i], rl_list[i])):
            if 'yes' == rrr:

                # if enable_ens:
                #     widx = wi * 2
                #     sub_yes = ens_res[widx].lower() == "yes"
                #     obj_yes = ens_res[widx + 1].lower() == "yes"

                #     if not (sub_yes and obj_yes):
                #         continue

                if dec != None:
                    mul = True

                dec = yyy
                dec_.append(yyy)

        if dec == None:
            dec = 'no_relation'
        elif mul == True:
            dec = '#multiple#'
        else:
            dec = dec

        trans.append(dec)
        trans_.append(dec_)

    # print(cnt)

    temp = r_list
    r_list = trans
    trans = temp

    golds = []
    preds = []

    for i in range(len(q_list)):
        q = q_list[i]
        rl = rl_list[i]
        r = r_list[i]

        if q in rl:
            assert q != 'no_relation'
            golds.append(q)
        else:
            golds.append('no_relation')

        preds.append(r)


    res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)
    print(len(golds))
    print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")
    print(f"& {(res['mean_P']):04.1f} & {res['std_P']:4.2f} & {(res['mean_R']):04.1f} & {res['std_R']:4.2f} & {(res['mean_F1']):04.1f} & {res['std_F1']:4.2f}")
    # sc = scorer.score(golds, preds, False)
    # print(f"{sc[0]*100:.2f}\t{sc[1]*100:.2f}\t{sc[2]*100:.2f}")
  

7 ************************** 
tacred_qwen_rpo_node_y_gradpo-gen_Qwen-Qwen3-4B-1shots_fs_tacred_test_episodes_1shots.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20260521.txt
150000
35.3 ± 1.36	24.5 ± 1.53	28.9 ± 1.46
& 35.3 & 1.36 & 24.5 & 1.53 & 28.9 & 1.46


In [ ]:
#### MV

cccnt = 0
all_p = []
all_g = []
for ii, ff in enumerate(files):
    source = ff
    # if not ff.startswith('Qwen') and not ff.startswith("meta-llama") and not ff.startswith("microsoft"):
    # if not (ff.startswith('meta-llama') or "gemma-2-9b-it" in ff or ff.startswith("microsoft")) :
    # if not ("Qwen-Qwen3-4B-5shots_fs_tacred_test_episodes_5shots_softmatch.pkl_query-0_is-summ-False_ep-start-0_ep-end-150000_20251202_hybrid.txt" in ff) :
    #     continue
    if "Qwen3" not in source or "no_ner" in source:
        continue
    print(f"{ii} ************************** ")
    cccnt += 1
    print(source)
    p_list = []
    q_list = []
    rl_list = []
    r_list = []

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            p_list.append(line)

        if '@@@@@@@@@@' in line:
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            q_list.append(line)

        if line == '$$$$$ query relation':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            rl_list.append(line)

        if line == '$$$$$ relation list':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            r_list.append(line)

        if line == '$$$$$ votes':
            ready = True
        else:
            ready = False


    for i in range(len(rl_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(rl_list[i])
        rl_list[i] = reconstructed_list

    for i in range(len(r_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(r_list[i])
        r_list[i] = reconstructed_list


    for i in range(len(p_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(p_list[i])
        p_list[i] = reconstructed_list

    # print(p_list[-1][0][0]['content'])

    cnt = 0
    trans = []
    trans_ = []
    for i in range(len(rl_list)):
        dec = None
        dec_ = []
        mul = False

        # ens_res = ens[i]

        for wi, (rrr, yyy) in enumerate (zip(r_list[i], rl_list[i])):
            if 'yes' == rrr:

                # if enable_ens:
                #     widx = wi * 2
                #     sub_yes = ens_res[widx].lower() == "yes"
                #     obj_yes = ens_res[widx + 1].lower() == "yes"

                #     if not (sub_yes and obj_yes):
                #         continue

                if dec != None:
                    mul = True

                dec = yyy
                dec_.append(yyy)

        if dec == None:
            dec = 'no_relation'
        elif mul == True:
            dec = '#multiple#'
        else:
            dec = dec

        trans.append(dec)
        trans_.append(dec_)

    # print(cnt)

    temp = r_list
    r_list = trans
    trans = temp

    golds = []
    preds = []

    for i in range(len(q_list)):
        q = q_list[i]
        rl = rl_list[i]
        r = r_list[i]

        if q in rl:
            assert q != 'no_relation'
            golds.append(q)
        else:
            golds.append('no_relation')

        preds.append(r)


    # res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)
    # print(len(golds))
    # print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")
    # print(f"& {(res['mean_P']):04.1f} & {res['std_P']:4.2f} & {(res['mean_R']):04.1f} & {res['std_R']:4.2f} & {(res['mean_F1']):04.1f} & {res['std_F1']:4.2f}")
    # sc = scorer.score(golds, preds, False)
    # print(f"{sc[0]*100:.2f}\t{sc[1]*100:.2f}\t{sc[2]*100:.2f}")

    all_p.append(preds)
    all_g.append(golds)
  

In [ ]:
from collections import defaultdict
mvs = defaultdict(lambda: defaultdict(int))

for ok in all_p:
    for ii, dd in enumerate(ok):
        mvs[ii][dd] += 1

best_list = [max(d, key=d.get) for d in mvs.values()]
print(len(best_list))


In [ ]:
res = evaluate_in_chunks(golds, best_list, scorer, chunk_size=30000)
print(len(golds))
print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "codes" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "codes" / "agents"))
import agent_scorer as scorer

source_dir = str(PROJECT_ROOT / "outputs" / "opt_prompt") + "/"
files = sorted(f for f in os.listdir(source_dir) if f.endswith(".txt"))
# print(files)
import numpy as np
def evaluate_in_chunks(golds, preds, scorer, chunk_size=30000):
    n_chunks  = (len(golds) // chunk_size) + 1
    precisions, recalls, f1s = [], [], []

    for i in range(n_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < n_chunks - 1 else len(golds)  # last chunk takes remainder

        g_chunk = golds[start:end]
        p_chunk = preds[start:end]

        if len(g_chunk) == 0:  # skip empty chunks
            continue

        sc = scorer.score(g_chunk, p_chunk, False)  # returns (P, R, F1)
        precisions.append(sc[0])
        recalls.append(sc[1])
        f1s.append(sc[2])

    precisions, recalls, f1s = np.array(precisions), np.array(recalls), np.array(f1s)

    # for p, r, f1 in zip(precisions, recalls, f1s):
    #     print(f"{p*100:.2f}\t{r*100:.2f}\t{f1*100:.2f}")

    results = {
        "mean_P": np.mean(precisions) * 100,
        "std_P": np.std(precisions) * 100,
        "mean_R": np.mean(recalls) * 100,
        "std_R": np.std(recalls) * 100,
        "mean_F1": np.mean(f1s) * 100,
        "std_F1": np.std(f1s) * 100,
    }

    return results

for f in files:
    print(f)

In [ ]:
cccnt = 0
for ii, ff in enumerate(files):
    source = ff
    # if not ff.startswith('Qwen') and not ff.startswith("meta-llama") and not ff.startswith("microsoft"):
    # if not (ff.startswith('meta-llama') or "gemma-2-9b-it" in ff or ff.startswith("microsoft")) :
    if not ("all_in_one" in ff) :
        continue
    print(f"{ii} ************************** ")
    cccnt += 1
    print(source)
    p_list = []
    q_list = []
    rl_list = []
    r_list = []

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            p_list.append(line)

        if '@@@@@@@@@@' in line:
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            q_list.append(line)

        if line == '$$$$$ query relation':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            rl_list.append(line)

        if line == '$$$$$ relation list':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            r_list.append(line)

        if line == '$$$$$ final decision':
            ready = True
        else:
            ready = False


    for i in range(len(rl_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(rl_list[i])
        rl_list[i] = reconstructed_list

    # for i in range(len(r_list)):
    #     # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
    #     #     print(rl_list[i])
    #     reconstructed_list = eval(r_list[i])
    #     r_list[i] = reconstructed_list


    # for i in range(len(p_list)):
    #     # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
    #     #     print(rl_list[i])
    #     reconstructed_list = eval(p_list[i])
    #     p_list[i] = reconstructed_list

    # print(p_list[-1][0][0]['content'])

    # cnt = 0
    # trans = []
    # trans_ = []
    # for i in range(len(rl_list)):
    #     dec = None
    #     dec_ = []
    #     mul = False

    #     # ens_res = ens[i]

    #     for wi, (rrr, yyy) in enumerate (zip(r_list[i], rl_list[i])):
    #         if 'yes' == rrr:

    #             # if enable_ens:
    #             #     widx = wi * 2
    #             #     sub_yes = ens_res[widx].lower() == "yes"
    #             #     obj_yes = ens_res[widx + 1].lower() == "yes"

    #             #     if not (sub_yes and obj_yes):
    #             #         continue

    #             if dec != None:
    #                 mul = True

    #             dec = yyy
    #             dec_.append(yyy)

    #     if dec == None:
    #         dec = 'no_relation'
    #     elif mul == True:
    #         dec = '#multiple#'
    #     else:
    #         dec = dec

    #     trans.append(dec)
    #     trans_.append(dec_)

    # # print(cnt)

    # temp = r_list
    # r_list = trans
    # trans = temp

    golds = []
    preds = []

    for i in range(len(q_list)):
        q = q_list[i]
        rl = rl_list[i]
        r = r_list[i]

        if q in rl:
            assert q != 'no_relation'
            golds.append(q)
        else:
            golds.append('no_relation')

        preds.append(r)


    res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)
    print(len(golds))
    print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")
    print(f"& {(res['mean_P']):04.1f} & {res['std_P']:4.2f} & {(res['mean_R']):04.1f} & {res['std_R']:4.2f} & {(res['mean_F1']):04.1f} & {res['std_F1']:4.2f}")
    # sc = scorer.score(golds, preds, False)
    # print(f"{sc[0]*100:.2f}\t{sc[1]*100:.2f}\t{sc[2]*100:.2f}")
  

In [ ]:
cccnt = 0
for ii, ff in enumerate(files):
    source = ff
    # if not ff.startswith('Qwen') and not ff.startswith("meta-llama") and not ff.startswith("microsoft"):
    # if not (ff.startswith('meta-llama') or "gemma-2-9b-it" in ff or ff.startswith("microsoft")) :
    if not ("fewrel" in ff) :
        continue
    print(f"{ii} ************************** ")
    cccnt += 1
    print(source)
    p_list = []
    q_list = []
    rl_list = []
    r_list = []

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            p_list.append(line)

        if '@@@@@@@@@@' in line:
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            q_list.append(line)

        if line == '$$$$$ query relation':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            rl_list.append(line)

        if line == '$$$$$ relation list':
            ready = True
        else:
            ready = False

    ready = False
    for line in open(os.path.join(source_dir, source), 'r').readlines():
        line = line.strip()
        if ready:
            r_list.append(line)

        if line == '$$$$$ votes':
            ready = True
        else:
            ready = False


    for i in range(len(rl_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(rl_list[i])
        rl_list[i] = reconstructed_list

    for i in range(len(r_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(r_list[i])
        r_list[i] = reconstructed_list


    for i in range(len(p_list)):
        # if "['org:top_members/employees', 'per:schools_attended', 'per:city_of_death', '" in rl_list[i]:
        #     print(rl_list[i])
        reconstructed_list = eval(p_list[i])
        p_list[i] = reconstructed_list

    # print(p_list[-1][0][0]['content'])

    cnt = 0
    trans = []
    trans_ = []
    for i in range(len(rl_list)):
        dec = None
        dec_ = []
        mul = False

        # ens_res = ens[i]

        for wi, (rrr, yyy) in enumerate (zip(r_list[i], rl_list[i])):
            if 'yes' == rrr:

                # if enable_ens:
                #     widx = wi * 2
                #     sub_yes = ens_res[widx].lower() == "yes"
                #     obj_yes = ens_res[widx + 1].lower() == "yes"

                #     if not (sub_yes and obj_yes):
                #         continue

                if dec != None:
                    mul = True

                dec = yyy
                dec_.append(yyy)

        if dec == None:
            dec = 'no_relation'
        elif mul == True:
            dec = '#multiple#'
        else:
            dec = dec

        trans.append(dec)
        trans_.append(dec_)

    # print(cnt)

    temp = r_list
    r_list = trans
    trans = temp

    golds = []
    preds = []

    for i in range(len(q_list)):
        q = q_list[i]
        rl = rl_list[i]
        r = r_list[i]

        if q in rl:
            assert q != 'no_relation'
            golds.append(q)
        else:
            golds.append('no_relation')

        preds.append(r)

    nos = 0
    for dd in golds:
        if dd == "no_relation":
            nos += 1

    print(len(golds))
    print(nos)
    # res = evaluate_in_chunks(golds, preds, scorer, chunk_size=30000)
    # print(len(golds))
    # print(f"{(res['mean_P']):04.1f} ± {res['std_P']:4.2f}\t{(res['mean_R']):04.1f} ± {res['std_R']:4.2f}\t{(res['mean_F1']):04.1f} ± {res['std_F1']:4.2f}")
    # print(f"& {(res['mean_P']):04.1f} & {res['std_P']:4.2f} & {(res['mean_R']):04.1f} & {res['std_R']:4.2f} & {(res['mean_F1']):04.1f} & {res['std_F1']:4.2f}")
    # sc = scorer.score(golds, preds, False)
    # print(f"{sc[0]*100:.2f}\t{sc[1]*100:.2f}\t{sc[2]*100:.2f}")
    break
  

In [ ]:
# tacred no_relation

150000
146138
0.974253333

150000
142503
0.95

In [ ]:
150000
146138